# trajectory-kit: Developer Guide

You're now in the engine room. This notebook covers everything below the
user-facing API:

1. The architecture — sim, registry, parsers, helpers
2. The domain registry — how parsers are wired in
3. The parser contract — five mandatory functions per parser
4. The `plan_shape` contract — what each request costs per atom per frame
5. The iterator helpers — `iter_records`, `iter_records_sample`
6. The query-help matchers — `_normalise_query_pair`, `_match`
7. The predicate-state pattern — why parsers precompute query state
8. End-to-end walkthrough — adding a new file format ("TOY" trajectory)

Read this if you're hacking on the parsers, debugging a query mismatch, or
adding support for a new file format.


## 0. Setup

In [1]:
import struct
import tempfile
from pathlib import Path
import numpy as np

# Synthetic files are written to a fresh temp directory. The path is printed
# below so you can browse to it and inspect the raw files if you want to.
tmp = Path(tempfile.mkdtemp(prefix="trajkit_tutorial_"))

# ── 20-atom synthetic system ──────────────────────────────────────────────
# Protein (PROT): ALA(6) + GLY(5) + PRO(3) = 14 atoms, 3 residues
# Solvent (SOLV): 2 x TIP3 water molecules = 6 atoms, 2 residues
#
# Bond graph (17 bonds):
#   Backbone: N-CA-C chain with peptide bonds
#   ALA sidechain: CA-CB
#   Carbonyl: C=O on each residue
#   PRO N closes the chain
#   Waters: each OH2 bonded to H1 and H2
#
# Degree distribution: 10 x degree-1, 6 x degree-2, 4 x degree-3
# Atom types used: NH1, H, CT1, CT3, C, O, CT2, N, CP1, OT, HT

pdb_text = """\
ATOM      1  N   ALA A   1       1.000   5.000   5.000  1.00  0.00      PROT
ATOM      2  HN  ALA A   1       0.100   5.800   5.300  1.00  0.00      PROT
ATOM      3  CA  ALA A   1       2.400   5.000   5.000  1.00  0.00      PROT
ATOM      4  CB  ALA A   1       3.000   6.300   5.200  1.00  0.00      PROT
ATOM      5  C   ALA A   1       3.200   3.900   4.600  1.00  0.00      PROT
ATOM      6  O   ALA A   1       2.800   2.800   4.400  1.00  0.00      PROT
ATOM      7  N   GLY A   2       4.600   4.100   4.500  1.00  0.00      PROT
ATOM      8  HN  GLY A   2       5.000   5.000   4.700  1.00  0.00      PROT
ATOM      9  CA  GLY A   2       5.800   3.200   4.100  1.00  0.00      PROT
ATOM     10  C   GLY A   2       7.200   3.500   4.000  1.00  0.00      PROT
ATOM     11  O   GLY A   2       7.700   4.600   4.200  1.00  0.00      PROT
ATOM     12  N   PRO A   3       8.100   2.500   3.700  1.00  0.00      PROT
ATOM     13  CA  PRO A   3       9.500   2.800   3.600  1.00  0.00      PROT
ATOM     14  C   PRO A   3      10.300   1.900   3.200  1.00  0.00      PROT
ATOM     15  OH2 TIP B   4      12.000   5.000   5.000  1.00  0.00      SOLV
ATOM     16  H1  TIP B   4      12.900   5.500   5.100  1.00  0.00      SOLV
ATOM     17  H2  TIP B   4      11.300   5.700   4.800  1.00  0.00      SOLV
ATOM     18  OH2 TIP B   5      14.000   3.000   5.500  1.00  0.00      SOLV
ATOM     19  H1  TIP B   5      14.800   3.600   5.400  1.00  0.00      SOLV
ATOM     20  H2  TIP B   5      13.400   3.700   5.200  1.00  0.00      SOLV
END
"""

psf_text = """\
PSF

       1 !NTITLE
 REMARKS synthetic 20-atom ALA-GLY-PRO + 2xTIP3

      20 !NATOM
       1 PROT     1 ALA  N    NH1     -0.47000     14.007           0
       2 PROT     1 ALA  HN   H        0.31000      1.008           0
       3 PROT     1 ALA  CA   CT1     -0.02000     12.011           0
       4 PROT     1 ALA  CB   CT3     -0.27000     12.011           0
       5 PROT     1 ALA  C    C        0.51000     12.011           0
       6 PROT     1 ALA  O    O       -0.51000     15.999           0
       7 PROT     2 GLY  N    NH1     -0.47000     14.007           0
       8 PROT     2 GLY  HN   H        0.31000      1.008           0
       9 PROT     2 GLY  CA   CT2     -0.18000     12.011           0
      10 PROT     2 GLY  C    C        0.51000     12.011           0
      11 PROT     2 GLY  O    O       -0.51000     15.999           0
      12 PROT     3 PRO  N    N       -0.29000     14.007           0
      13 PROT     3 PRO  CA   CP1     -0.02000     12.011           0
      14 PROT     3 PRO  C    C        0.51000     12.011           0
      15 SOLV     4 TIP  OH2  OT      -0.83400     15.999           0
      16 SOLV     4 TIP  H1   HT       0.41700      1.008           0
      17 SOLV     4 TIP  H2   HT       0.41700      1.008           0
      18 SOLV     5 TIP  OH2  OT      -0.83400     15.999           0
      19 SOLV     5 TIP  H1   HT       0.41700      1.008           0
      20 SOLV     5 TIP  H2   HT       0.41700      1.008           0

      17 !NBOND: bonds
       1       2       1       3       3       4       3       5
       5       6       5       7       7       8       7       9
       9      10      10      11      10      12      12      13
      13      14      15      16      15      17      18      19
      18      20

       0 !NTHETA: angles

       0 !NPHI: dihedrals

       0 !NIMPHI: impropers

"""

def write_dcd(path, n_atoms=20, n_frames=5):
    """5-frame DCD. Each frame shifts all atoms by +1 A along x."""
    def record(payload):
        n = struct.pack('<i', len(payload))
        return n + payload + n
    icntrl = [0] * 20
    icntrl[0] = n_frames
    hdr = b'CORD' + struct.pack('<20i', *icntrl)
    hdr += b'\x00' * (84 - len(hdr))
    title = struct.pack('<i', 1) + b'synthetic 20-atom trajectory    '
    natom = struct.pack('<i', n_atoms)
    x0 = np.array([ 1.0, 0.1, 2.4, 3.0, 3.2, 2.8,
                    4.6, 5.0, 5.8, 7.2, 7.7, 8.1,
                    9.5,10.3,12.0,12.9,11.3,14.0,14.8,13.4], dtype=np.float32)
    y0 = np.array([ 5.0, 5.8, 5.0, 6.3, 3.9, 2.8,
                    4.1, 5.0, 3.2, 3.5, 4.6, 2.5,
                    2.8, 1.9, 5.0, 5.5, 5.7, 3.0, 3.6, 3.7], dtype=np.float32)
    z0 = np.array([ 5.0, 5.3, 5.0, 5.2, 4.6, 4.4,
                    4.5, 4.7, 4.1, 4.0, 4.2, 3.7,
                    3.6, 3.2, 5.0, 5.1, 4.8, 5.5, 5.4, 5.2], dtype=np.float32)
    with open(path, 'wb') as f:
        f.write(record(hdr))
        f.write(record(title))
        f.write(record(natom))
        for fi in range(n_frames):
            xs = x0 + float(fi)
            f.write(record(xs.tobytes()))
            f.write(record(y0.tobytes()))
            f.write(record(z0.tobytes()))

pdb_path = tmp / 'synthetic.pdb'
psf_path = tmp / 'synthetic.psf'
dcd_path = tmp / 'synthetic.dcd'

pdb_path.write_text(pdb_text, encoding='ascii')
psf_path.write_text(psf_text, encoding='ascii')
write_dcd(dcd_path)

print('Synthetic files written to:', tmp)
print('  PDB: 20 atoms (ALA-GLY-PRO backbone + 2x TIP3 water)')
print('  PSF: 20 atoms, 17 bonds, 5 residues, 2 segments (PROT/SOLV)')
print('  DCD: 5 frames - x coordinates shift +1 A per frame')


Synthetic files written to: /var/folders/b7/cnz6gkn92hl85yrrz4d00lgm0000gn/T/trajkit_tutorial_qw6eiwh7
  PDB: 20 atoms (ALA-GLY-PRO backbone + 2x TIP3 water)
  PSF: 20 atoms, 17 bonds, 5 residues, 2 segments (PROT/SOLV)
  DCD: 5 frames - x coordinates shift +1 A per frame


In [2]:
from trajectory_kit import sim

s = sim(
    typing=pdb_path,
    topology=psf_path,
    trajectory=dcd_path,
)


## 1. Architecture overview

`trajectory-kit` is a thin orchestration layer over a set of single-format
parser modules. The layout, top to bottom:

```
                 sim (main.py)
                  |
         _domain_registry            <- function-name templates per domain/format
                  |
          parser modules             <- pdb_parse, psf_parse, dcd_parse, ...
                  |
   _file_parse_help, _query_help     <- shared iterator + matcher helpers
```

Adding a new format means writing one parser module that satisfies the
contract, then adding an entry to `_domain_registry`.


## 2. The domain registry

Look at the registry directly. There are three domain entries — one each
for typing, topology, trajectory. Each entry lists the supported file
extensions and the *templates* used to resolve parser function names.


In [3]:
import pprint
pprint.pp(s._domain_registry)


{'typing': {'supported_formats': {'.xyz', '.pdb', '.mae'},
            'file_attr': 'type_file',
            'type_attr': 'type_type',
            'keys_attr': 'type_file_keys',
            'reqs_attr': 'type_file_reqs',
            'keys_fn_template': '_get_type_keys_reqs_{fmt}',
            'plan_fn_template': '_plan_type_query_{fmt}',
            'plan_shape_fn_template': '_get_type_plan_shape_{fmt}',
            'query_fn_template': '_get_type_query_{fmt}',
            'update_fn_template': '_update_type_globals_{fmt}',
            'label': 'typing'},
 'topology': {'supported_formats': {'.psf', '.mae'},
              'file_attr': 'top_file',
              'type_attr': 'top_type',
              'keys_attr': 'topo_file_keys',
              'reqs_attr': 'topo_file_reqs',
              'keys_fn_template': '_get_topology_keys_reqs_{fmt}',
              'plan_fn_template': '_plan_topology_query_{fmt}',
              'plan_shape_fn_template': '_get_topology_plan_shape_{fmt}',
            

The template fields:

| Template | Format | Returns |
|----------|--------|---------|
| `keys_fn_template` | `_get_<domain>_keys_reqs_<fmt>` | `(set_of_keywords, set_of_requests)` for the file. |
| `plan_fn_template` | `_plan_<domain>_query_<fmt>` | A raw plan dict for a query+request. |
| `plan_shape_fn_template` | `_get_<domain>_plan_shape_<fmt>` | `(output_kind, trailing_shape, bytes_per_match)` for a request. |
| `query_fn_template` | `_get_<domain>_query_<fmt>` | The actual data for a query+request. |
| `update_fn_template` | `_update_<domain>_globals_<fmt>` | A dict of global facts (atom count, etc.) extracted from the file. |

`<domain>` is `type` (typing), `topology`, or `trajectory`. `<fmt>` is the
file extension without the dot. All five functions must exist on a parser
module — `_validate_domain_module_contract` checks this at first import.


## 3. The parser contract

Every parser module must export these five functions for each domain it
serves:

### `_get_<domain>_keys_reqs_<fmt>(filepath)`

Return `(set_of_keywords, set_of_requests)`. Keywords are valid query keys;
requests are valid request strings. Called once at file load.

### `_plan_<domain>_query_<fmt>(filepath, query_dictionary, request_string, keywords_available, requests_available)`

Return a raw plan dict. Required keys: `planner_mode` (`"header"` or
`"stochastic"`), `n_atoms`, `n_frames`. May include `confidence`,
`sampling`-related fields, format-specific extras. Or
`{'planner_mode': ..., 'supported': False, 'reason': ...}` to opt out.

### `_get_<domain>_plan_shape_<fmt>(request_string)`

Return `(output_kind, trailing_shape, bytes_per_match)`. Pure function — no
file I/O, depends only on the request string.

| `output_kind` | Meaning | `bytes_per_match` |
|---------------|---------|-------------------|
| `"per_atom"` | One value per atom (e.g. `atom_names`, `charges`). | size in bytes |
| `"per_atom_per_frame"` | Per-atom-per-frame value (e.g. `positions`). | size in bytes |
| `"scalar_property"` | One value per file (e.g. `property-box_size`). | `None` |
| `"selector"` | An id list — used internally for selection. | `None` |

When `bytes_per_match is None` the planner is short-circuited centrally
(returns `None` from `_plan_domain_request`); the parser's planner is never
called for those requests.

### `_get_<domain>_query_<fmt>(filepath, query_dictionary, request_string, keywords_available, requests_available)`

Execute the query. Returns the bare value: a list, scalar, tuple, ndarray,
or None. **No metadata wrapping.** Per-atom array payloads (`positions`,
`velocities`) return ndarrays of shape `(n_frames, n_atoms_selected, 3)`.

### `_update_<domain>_globals_<fmt>(filepath)`

Return a dict of system-level facts (`num_atoms`, `num_frames`,
`num_residues`, `start_box_size`, etc.) — anything that should populate
`sim.global_system_properties`. Empty dict if nothing applies.


## 4. The `plan_shape` contract — concrete examples

`plan_shape` is the simplest contract function: pure, no I/O, depends only
on the request string. It tells the standardiser what each request costs
without touching the file.


In [4]:
from trajectory_kit import pdb_parse, psf_parse, dcd_parse, mae_parse

# PDB typing (per-atom data)
print('pdb global_ids   :', pdb_parse._get_type_plan_shape_pdb('global_ids'))
print('pdb atom_names   :', pdb_parse._get_type_plan_shape_pdb('atom_names'))
print('pdb positions    :', pdb_parse._get_type_plan_shape_pdb('positions'))
print('pdb prop-natoms  :', pdb_parse._get_type_plan_shape_pdb('property-number_of_atoms'))
print()

# PSF topology
print('psf charges      :', psf_parse._get_topology_plan_shape_psf('charges'))
print('psf atom_types   :', psf_parse._get_topology_plan_shape_psf('atom_types'))
print()

# DCD trajectory
print('dcd positions    :', dcd_parse._get_trajectory_plan_shape_dcd('positions'))
print('dcd global_ids   :', dcd_parse._get_trajectory_plan_shape_dcd('global_ids'))


pdb global_ids   : ('per_atom', (), 8)
pdb atom_names   : ('per_atom', (), 16)
pdb positions    : ('per_atom', (3,), 12)
pdb prop-natoms  : ('scalar_property', (), None)

psf charges      : ('per_atom', (), 8)
psf atom_types   : ('per_atom', (), 16)

dcd positions    : ('per_atom_per_frame', (3,), 12)
dcd global_ids   : ('selector', (), None)


Note `dcd global_ids` returns `output_kind="selector"` with
`bytes_per_match=None`. DCD has no per-atom data to filter on; the
`global_ids` request exists only as a selection-time signal returning
`None` (no constraint contributed by the trajectory domain).


## 5. The iterator helpers

All text-based parsers (PDB, PSF, XYZ) share two iterator functions in
`_file_parse_help`. They abstract away the line-by-line plumbing so each
parser only writes record predicates and row parsers.


In [5]:
from trajectory_kit import _file_parse_help as fph

# Predicate mode: yield every line matching a predicate.
# Used by PDB / XYZ where atom records are scattered through the file.

def parse_pdb_atom(line, idx):
    # idx is the 0-based atom index, supplied by the iterator.
    return {
        'global_id': idx,
        'atom_name': line[12:16].strip(),
        'residue':   line[17:21].strip(),
    }

atoms = list(fph.iter_records(
    pdb_path,
    mode='predicate',
    record_pred=lambda line: line[0:6] in ('ATOM  ', 'HETATM'),
    parse_row=parse_pdb_atom,
))

print('first 3 atoms:', atoms[:3])
print('total atoms  :', len(atoms))


first 3 atoms: [{'global_id': 0, 'atom_name': 'N', 'residue': 'ALA'}, {'global_id': 1, 'atom_name': 'HN', 'residue': 'ALA'}, {'global_id': 2, 'atom_name': 'CA', 'residue': 'ALA'}]
total atoms  : 20


In [6]:
# Counted mode: find a header line, read exactly N following lines.
# Used by PSF where !NATOM declares the count up-front.

def parse_psf_atom(row, idx):
    fields = row.split()
    return {
        'global_id':  idx,
        'segment':    fields[1],
        'residue_id': int(fields[2]),
        'atom_type':  fields[5],
        'charge':     float(fields[6]),
    }

psf_atoms = list(fph.iter_records(
    psf_path,
    mode='counted',
    header_pred=lambda line: line.strip().endswith('!NATOM'),
    count_from_header=lambda line: int(line.split()[0]),
    parse_row=parse_psf_atom,
))

print('first PSF atom:', psf_atoms[0])


first PSF atom: {'global_id': 0, 'segment': 'PROT', 'residue_id': 1, 'atom_type': 'NH1', 'charge': -0.47}


### `iter_records_sample` — what powers the stochastic planner

When a file is too large to scan exhaustively for planning, the planner
uses a single-pass Bernoulli sampler. It reads the file once, samples
records with a probability tuned to hit a target sample size, and returns
both the sampled records and counts of total / sampled / eligible lines.


In [7]:
sample_info = fph.iter_records_sample(
    pdb_path,
    record_pred=lambda line: line[0:6] in ('ATOM  ', 'HETATM'),
    parse_row=parse_pdb_atom,
    target_sample_size=3000,
    rng_seed=42,
)

print('total lines              :', sample_info['number_of_lines'])
print('sampled lines            :', sample_info['number_of_sampled_lines'])
print('sampled eligible records :', sample_info['number_of_sampled_eligible_records'])
print('records returned         :', len(sample_info['sampled_records']))
print('first sampled record     :', sample_info['sampled_records'][0])


total lines              : 21
sampled lines            : 21
sampled eligible records : 20
records returned         : 20
first sampled record     : {'global_id': 0, 'atom_name': 'N', 'residue': 'ALA'}


For our 20-atom file `target_sample_size=3000` exceeds the file size, so
sampling probability is 1 — every line is read. On a 10M-line file the
sampler would visit only the lines whose physical index matches a geometric
skip schedule, keeping the planner cheap.


## 6. The query-help matchers

`_query_help` provides the small set of matchers every parser uses to
evaluate query predicates. Key entry points:

- `_normalise_query_pair(value, range_style=False)` — turns a raw query
  value into a normalised `(include, exclude)` pair regardless of input
  style (bare scalar, set, list, tuple, or explicit pair wrapper).
- `_match(value, include_set, exclude_set)` for categorical fields.
- `_match_range_scalar(value, include_range, exclude_range)` for numeric
  fields.
- `_normalise_bonded_with_pair(value)` — turns any accepted shape (single
  dict, list of dicts, `(include, exclude)` tuple, or `None`) into a
  canonical `(include_blocks, exclude_blocks)` pair.
- `_freeze_query(obj)` — recursively converts a query dict into a
  hashable, deterministically-ordered tuple for use as a cache key.
- `MAX_BONDED_WITH_DEPTH` — constant (16) capping recursion for nested
  `bonded_with` neighbour queries.

**A note for new parsers:** when you add a topology format that supports
bond-graph filtering, use `_normalise_bonded_with_pair` to accept the
user-facing shorthand forms uniformly, and use `_freeze_query` as your
cache key builder. You do not need to re-implement the shorthand handling
or write your own cache-key function.


In [8]:
from trajectory_kit import _query_help as qh

# Categorical: bare value, set, or (include, exclude) tuple all normalise
# to the same shape.
print(qh._normalise_query_pair('CA'))
print(qh._normalise_query_pair({'CA', 'CB'}))
print(qh._normalise_query_pair(({'CA'}, {'HN'})))

# Range vs membership: tuple = inclusive range, list = membership.
print(qh._normalise_query_pair((1, 5),                range_style=True))
print(qh._normalise_query_pair([1, 5],                range_style=True))
print(qh._normalise_query_pair(((1, 10), (3, 5)),     range_style=True))


({'CA'}, set())
({'CB', 'CA'}, set())
({'CA'}, {'HN'})
(((1, 5),), ())
(((1, 1), (5, 5)), ())
(((1, 10),), ((3, 5),))


In [9]:
# bonded_with shorthand all normalises to the canonical (inc, exc) form.
print(qh._normalise_bonded_with_pair(None))
print(qh._normalise_bonded_with_pair({'total': True, 'count': {'eq': 1}}))
print(qh._normalise_bonded_with_pair([{'total': True, 'count': {'eq': 1}}]))
print(qh._normalise_bonded_with_pair(
    ([{'total': True, 'count': {'eq': 1}}],
     [{'total': True, 'count': {'eq': 0}}])
))

# Deterministic freezing — same semantic content, different insertion order,
# same frozen key.
q1 = {'atom_type': 'OT', 'count': {'ge': 1}}
q2 = {'count': {'ge': 1}, 'atom_type': 'OT'}
print(qh._freeze_query(q1) == qh._freeze_query(q2))


([], [])
([{'total': True, 'count': {'eq': 1}}], [])
([{'total': True, 'count': {'eq': 1}}], [])
([{'total': True, 'count': {'eq': 1}}], [{'total': True, 'count': {'eq': 0}}])
True


In [10]:
# Match a value against the normalised pair.
inc, exc = qh._normalise_query_pair(({'CA', 'CB'}, {'CA'}))
print(qh._match('CA', inc, exc))   # False - in include but also in exclude
print(qh._match('CB', inc, exc))   # True
print(qh._match('HN', inc, exc))   # False - not in include


False
True
False


## 7. The predicate-state pattern

Parsers don't call `_normalise_query_pair` on every atom — that would be
wasteful. Instead they precompute a **predicate state** dict once per
query, then the per-atom matcher reads from that dict.

The state carries both the normalised include/exclude pairs *and* `need_*`
flags so the matcher can skip checks for fields the query doesn't constrain.


In [11]:
from trajectory_kit.pdb_parse import _get_pdb_type_predicate_state

state = _get_pdb_type_predicate_state({
    'atom_name':   'CA',
    'residue_ids': (1, 2),
})

# Show only the active flags + populated entries
active = {k: v for k, v in state.items() if k.startswith('need_') and v}
print('Active predicate flags:', active)

print('atom_inc:', state['atom_inc'])
print('atom_exc:', state['atom_exc'])
print('ri_inc  :', state['ri_inc'])
print('ri_exc  :', state['ri_exc'])


Active predicate flags: {'need_atom': True, 'need_li': True, 'need_ri': True, 'need_occ': True, 'need_temp': True, 'need_x': True, 'need_y': True, 'need_z': True}
atom_inc: {'CA'}
atom_exc: set()
ri_inc  : ((1, 2),)
ri_exc  : ()


The shared predicate state is the reason the planner and the executor
agree exactly on which atoms a query selects: both call
`_get_pdb_type_predicate_state` and `_pdb_atom_matches_query` on the same
inputs. There's no separate "fast path" predicate that could drift from
the real one.


## 8. End-to-end walkthrough — adding a new file format

We'll add a minimal fictitious trajectory format called **TOY**: a plain
text format with one frame per file, listing `n_atoms`, then `x y z` per
atom on subsequent lines.

```
3
0.0 0.0 0.0
1.0 0.0 0.0
2.0 0.0 0.0
```

This walkthrough shows every piece slotting together. In practice you'd
save the parser as `toy_parse.py` in the package and add the registry
entry to `main.py`. For the demo we'll do everything in this notebook.


### Step 1. Write a sample TOY file

In [12]:
toy_path = tmp / 'sample.toy'
toy_path.write_text('''3
0.0 0.0 0.0
1.0 0.0 0.0
2.0 0.0 0.0
''')

print('toy file at:', toy_path)
print(toy_path.read_text())


toy file at: /var/folders/b7/cnz6gkn92hl85yrrz4d00lgm0000gn/T/trajkit_tutorial_qw6eiwh7/sample.toy
3
0.0 0.0 0.0
1.0 0.0 0.0
2.0 0.0 0.0



### Step 2. Write the parser module

Five contract functions for the trajectory domain. We'll define them in this
cell as plain functions, then bundle them into a fake module so the registry
can find them.


In [13]:
import types
import numpy as np

# ----- contract function 1: _update_trajectory_globals_toy -----------------
def _update_trajectory_globals_toy(filepath):
    with open(filepath, 'r') as f:
        n = int(f.readline().strip())
    return {'num_atoms': n, 'num_frames': 1}


# ----- contract function 2: _get_trajectory_keys_reqs_toy ------------------
def _get_trajectory_keys_reqs_toy(filepath):
    keywords = {'global_ids'}
    requests = {'positions', 'global_ids'}
    return keywords, requests


# ----- contract function 3: _get_trajectory_plan_shape_toy -----------------
def _get_trajectory_plan_shape_toy(request_string):
    match request_string:
        case 'positions':  return ('per_atom_per_frame', (3,), 12)
        case 'global_ids': return ('selector', (), None)
        case _:
            raise ValueError(f'Unsupported TOY request: {request_string!r}')


# ----- contract function 4: _plan_trajectory_query_toy ---------------------
def _plan_trajectory_query_toy(filepath, query_dictionary, request_string,
                                keywords_available, requests_available):
    if request_string not in requests_available:
        raise ValueError(f'Unsupported request: {request_string!r}')
    with open(filepath, 'r') as f:
        n = int(f.readline().strip())
    return {
        'planner_mode': 'header',
        'n_atoms':      n,
        'n_frames':     1,
    }


# ----- contract function 5: _get_trajectory_query_toy ----------------------
def _get_trajectory_query_toy(filepath, query_dictionary, request_string,
                                keywords_available, requests_available):
    if request_string == 'global_ids':
        return None    # TOY has no per-atom data to filter on

    if request_string == 'positions':
        with open(filepath, 'r') as f:
            n = int(f.readline().strip())
            coords = np.empty((n, 3), dtype=np.float32)
            for i in range(n):
                coords[i] = [float(x) for x in f.readline().split()]
        # Apply the global_ids selection from the (selected_globals, set()) tuple
        gids = query_dictionary['global_ids'][0]
        return coords[list(gids)][np.newaxis, :, :]    # (1, n_selected, 3)

    raise ValueError(f'Unsupported request: {request_string!r}')


# Bundle into a module-like object so importlib lookups would resolve it.
toy_parse = types.ModuleType('trajectory_kit.toy_parse')
for name, fn in [
    ('_update_trajectory_globals_toy', _update_trajectory_globals_toy),
    ('_get_trajectory_keys_reqs_toy',  _get_trajectory_keys_reqs_toy),
    ('_get_trajectory_plan_shape_toy', _get_trajectory_plan_shape_toy),
    ('_plan_trajectory_query_toy',     _plan_trajectory_query_toy),
    ('_get_trajectory_query_toy',      _get_trajectory_query_toy),
]:
    setattr(toy_parse, name, fn)

print('toy_parse module assembled')


toy_parse module assembled


### Step 3. Register the format with `sim`

We extend the existing trajectory entry in `_domain_registry` to claim the
`.toy` extension, and stuff our fake module into the parse module cache so
`sim` doesn't try to `importlib.import_module('trajectory_kit.toy_parse')`.

In a real package you would simply add `'.toy'` to the trajectory registry's
`supported_formats` set in `main.py`, and place `toy_parse.py` next to
`dcd_parse.py`.


In [14]:
# Fresh sim with no files yet so we can wire the format before loading.
s_toy = sim()

# Register the extension.
s_toy._domain_registry['trajectory']['supported_formats'].add('.toy')

# Inject our in-notebook module into the cache so the registry will find it.
s_toy._module_cache['toy'] = toy_parse

# Validate the contract on the fly (sim does this automatically on import,
# but we're sneaking the module in so let's invoke it explicitly).
s_toy._validate_domain_module_contract('trajectory', 'toy', toy_parse)
print('contract OK')


contract OK


### Step 4. Load and use the TOY file

Note: `positions()` requires a typing or topology file (or an explicit
`TYPE_Q` / `TOPO_Q`) to define the atom selection. With only a trajectory
loaded, use `fetch()` and pass the selection through `TRAJ_Q`.


In [15]:
s_toy.load_trajectory(toy_path)

print('Loaded:', s_toy.traj_file)
print('Type  :', s_toy.traj_type)
print('Keys  :', sorted(s_toy.get_traj_keys()))
print('Reqs  :', sorted(s_toy.get_traj_reqs()))


Loaded: /var/folders/b7/cnz6gkn92hl85yrrz4d00lgm0000gn/T/trajkit_tutorial_qw6eiwh7/sample.toy
Type  : .toy
Keys  : ['global_ids']
Reqs  : ['global_ids', 'positions']


In [16]:
# Read positions for all 3 atoms in the toy file.
result = s_toy.fetch(
    TRAJ_Q={'global_ids': (list(range(3)), set())},
    TRAJ_R='positions',
)
print('positions payload:')
print(result['trajectory'])
print('shape:', result['trajectory'].shape)


positions payload:
[[[0. 0. 0.]
  [1. 0. 0.]
  [2. 0. 0.]]]
shape: (1, 3, 3)


In [17]:
# Use devFlag to see the planner output for the new format.
import pprint

out = s_toy.fetch(
    TRAJ_Q={'global_ids': (list(range(3)), set())},
    TRAJ_R='positions',
    devFlag=True,
)
pprint.pp(out['plan'])


{'trajectory': {'planner_mode': 'header',
                'source': 'trajectory',
                'file_format': 'toy',
                'request': 'positions',
                'n_atoms': 3,
                'n_frames': 1,
                'bytes_per_atom_per_frame': 12,
                'estimated_bytes': 36},
 'combined': {'n_atoms_upper_bound': 3, 'total_estimated_bytes': 36}}


That's the whole flow. The toy parser now participates in the same
planner / standardiser / envelope machinery as the built-in formats, and
the user-facing `fetch()` works against `.toy` files exactly as it does
against `.dcd` files.


## 9. Closing notes

- **Contract validation** runs once per format, on the first
  `_get_parse_module(fmt)` call. If your parser is missing a function,
  you'll get a clear `AttributeError` listing the missing names — not a
  cryptic failure deep in `_execute_domain_request`.
- **Hard-fail policy.** The standardiser hard-fails if a required tier-1
  plan field (`n_atoms`, `n_frames`) is missing or if `bytes_per_match`
  comes back `None` for a non-property request. These errors carry the
  parser name and request string so the offending function is obvious.
- **No metadata tuples.** Parsers never wrap return values in
  `(value, meta)` tuples. File-level metadata is built independently from
  the `_update_*_globals_*` functions and surfaced via the envelope's
  `metadata` block.
- **Read existing parsers first.** `pdb_parse.py` (predicate mode +
  stochastic planner), `dcd_parse.py` (header planner, binary), and
  `coor_parse.py` (single-frame trajectory) span most of the patterns
  you'll need. Cribbing from them is faster than starting from scratch.
